### LCEL 개요

LCEL은 기존의 체인을 Python 코드로 복잡하게 구성하던 방식에서 벗어나, |(파이프) 연산자를 사용해 데이터 흐름을 쉽게 구성할 수 있게 해줍니다. 유닉스 파이프라인처럼 작동

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [2]:
template ="{product} 제품 홍보문구 작성해줘"
prompt = PromptTemplate( input_variables=['product'], template=template )

gpt = ChatOpenAI( model="gpt-3.5-turbo") 
chain = prompt | gpt
rst = chain.invoke( {'product': '컴퓨터'} )
print(rst.content)

# rst =gpt.invoke( prompt.format(product="컴퓨터") )
# print( rst.content)

"최신기술을 담은 혁신적인 컴퓨팅 경험을 만나보세요! 성능과 디자인이 어우러진 우수한 제품으로 여러분의 일상을 더욱 편리하고 효율적으로 만들어드립니다. 지금 당장 만나보세요!"


In [ ]:
template ="{product} 제품 홍보문구 작성해줘"
prompt = PromptTemplate( input_variables=['product'], template=template )

outputParser = StrOutputParser() # 문자열 출력

gpt = ChatOpenAI( model="gpt-3.5-turbo") 
# chain = prompt | gpt | outputParser : 입력 -> 모델 -> 출력 결과
chain = prompt | gpt | StrOutputParser() # 답변이 문자열로 바로 출력, prompt -> gpt -> outputParser
rst = chain.invoke( {'product': '마우스'} )
print(rst)

"최고의 성능과 편의성을 자랑하는 마우스! 탁월한 반응 속도로 게임을 더욱 즐기고, 스마트한 디자인으로 업무 효율을 높이는 데 최적화된 마우스를 만나보세요. 혁신적인 기술력으로 더 나은 사용자 경험을 제공하는 마우스로 여러분을 사로잡을 것입니다. 지금 당장 만나보세요!"


In [4]:
from langchain_core.output_parsers import JsonOutputParser
from pydantic import BaseModel, Field

class Country(BaseModel):
    continet: str = Field(description="사용자가 물어본 나라가 속한 대륙")
    population: int = Field(description="사용자가 물어본 나라의 인구")  

parser = JsonOutputParser(pydantic_object=Country)
format_instructions = parser.get_format_instructions()


template = "answer the question.\n{question}\n\n{format_instructions}"
prompt = PromptTemplate(   template=template,
    input_variables=["question"],   
    partial_variables={"format_instructions": format_instructions}
)

gpt = ChatOpenAI( model_name="gpt-3.5-turbo", temperature=0)
chain = prompt | gpt | parser
rst = chain.invoke( {'question': "아르헨티나는 어떤 나라야?"} )
print(rst)

# rst = gpt.invoke( prompt.format( question="아르헨티나는 어떤 나라야?" ) )
# print( rst.content )

{'continet': '남아메리카', 'population': 45195774}


## 3분 퀴즈: "비트코인은 언제 개발됐어?" lcel 을 이용하여 답변을 얻으시요( 출력형식:DatetimeOutputParser)

In [8]:
from langchain.output_parsers import DatetimeOutputParser

template = "Answer the question: {question}\nformat은 다음과 같습니다.]n{format_instructions}"
output_parser = DatetimeOutputParser()
format_instructions = output_parser.get_format_instructions()
prompt = PromptTemplate(
    template=template,
    input_variables=['question'],
    partial_variables={'format_instructions': format_instructions}
)
gpt = ChatOpenAI(model='gpt-3.5-turbo', temperature=0)
chain = prompt | gpt | output_parser
rst = chain.invoke({'question':"비트코인은 언제 개발됐어?"})
print(rst)

2009-01-03 18:15:05
